<a href="https://colab.research.google.com/github/JaredOzarzak/biomechanics-analysis-pipeline/blob/main/NBA_Play_by_Play_Data_Auditor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NBA Play-by-Play Data Auditor
**Author:** Jared Ozarzak
**Focus:** Basketball Operations, Data Validation, & Stats Auditing

This notebook serves as a programmatic data validation tool designed to audit live NBA play-by-play (PBP) data feeds. It systematically scans raw tracking data to identify and log statistical discrepancies, sequence anomalies, and attribution errors in real-time.

In [4]:
import pandas as pd
import io

# Formatting output to look like a clean terminal report
def print_header(title):
    print(f"\n{'='*40}")
    print(f"{title}")
    print(f"{'='*40}")

## Step 1: Generate the Raw Data Feed
In a live environment, this data would stream directly from the arena's optical tracking and stat-entry systems. For this demonstration, we are generating a mock dataset containing intentional data entry errors (e.g., impossible point values, missing player IDs, and illogical event sequences).

In [5]:
# Simulating a live CSV data feed from a Cleveland Cavaliers game
csv_data = """QUARTER,TIME_REMAINING,EVENT_TYPE,PLAYER_ID,POINTS_SCORED
1,11:45,Made Field Goal,1001,2
1,11:42,Rebound,1002,0
1,10:30,Turnover,,0
1,09:15,Free Throw,1003,2
1,08:44,Missed Field Goal,1004,0
1,08:42,Rebound,1005,0"""

# Writing the simulated feed to a local file
with open('live_game_feed.csv', 'w') as f:
    f.write(csv_data)

print("System: Mock data feed generated successfully.")

System: Mock data feed generated successfully.


## Step 2: Ingest and Inspect the Data
We utilize `pandas` to load the live data feed into a DataFrame for rapid processing and analysis.

In [6]:
# Load the data
df = pd.read_csv('live_game_feed.csv')

# Display the raw feed to the user
print_header("RAW DATA FEED INGESTED")
display(df)


RAW DATA FEED INGESTED


,QUARTER,TIME_REMAINING,EVENT_TYPE,PLAYER_ID,POINTS_SCORED
0,1,11:45,Made Field Goal,1001.0,2
1,1,11:42,Rebound,1002.0,0
2,1,10:30,Turnover,NaN,0
3,1,09:15,Free Throw,1003.0,2
4,1,08:44,Missed Field Goal,1004.0,0
5,1,08:42,Rebound,1005.0,0


## Step 3: Define the Audit Logic
We establish three primary validation checks based on official NBA scorekeeping rules:
1. **Scoring Validation:** Free throws cannot exceed 1 point.
2. **Sequence Validation:** A rebound cannot be credited immediately after a made field goal (unless a free throw is pending, which is simplified here).
3. **Attribution Validation:** Critical events like turnovers must be attributed to a specific player ID.

In [7]:
def audit_scoring_values(df):
    errors = 0
    ft_anomalies = df[(df['EVENT_TYPE'] == 'Free Throw') & (df['POINTS_SCORED'] > 1)]
    for idx, row in ft_anomalies.iterrows():
        print(f"[SCORING ERROR] Q{row['QUARTER']} {row['TIME_REMAINING']} - Free Throw logged as {row['POINTS_SCORED']} points (Player: {row['PLAYER_ID']})")
        errors += 1
    return errors

def audit_rebound_logic(df):
    errors = 0
    for i in range(1, len(df)):
        current_event = df.loc[i, 'EVENT_TYPE']
        prev_event = df.loc[i-1, 'EVENT_TYPE']
        if current_event == 'Rebound' and prev_event == 'Made Field Goal':
            print(f"[SEQUENCE ERROR] Q{df.loc[i, 'QUARTER']} {df.loc[i, 'TIME_REMAINING']} - Rebound logged immediately after a Made Field Goal.")
            errors += 1
    return errors

def audit_missing_attributions(df):
    errors = 0
    missing_data = df[(df['EVENT_TYPE'].isin(['Turnover', 'Foul'])) & (df['PLAYER_ID'].isnull())]
    for idx, row in missing_data.iterrows():
        print(f"[ATTRIBUTION ERROR] Q{row['QUARTER']} {row['TIME_REMAINING']} - {row['EVENT_TYPE']} is missing a Player ID.")
        errors += 1
    return errors

def audit_scoring_values(df):
    errors = 0
    ft_anomalies = df[(df['EVENT_TYPE'] == 'Free Throw') & (df['POINTS_SCORED'] > 1)]
    for idx, row in ft_anomalies.iterrows():
        print(f"[SCORING ERROR] Q{row['QUARTER']} {row['TIME_REMAINING']} - Free Throw logged as {row['POINTS_SCORED']} points (Player: {row['PLAYER_ID']})")
        errors += 1
    return errors

def audit_rebound_logic(df):
    errors = 0
    for i in range(1, len(df)):
        current_event = df.loc[i, 'EVENT_TYPE']
        prev_event = df.loc[i-1, 'EVENT_TYPE']
        if current_event == 'Rebound' and prev_event == 'Made Field Goal':
            print(f"[SEQUENCE ERROR] Q{df.loc[i, 'QUARTER']} {df.loc[i, 'TIME_REMAINING']} - Rebound logged immediately after a Made Field Goal.")
            errors += 1
    return errors

def audit_missing_attributions(df):
    errors = 0
    missing_data = df[(df['EVENT_TYPE'].isin(['Turnover', 'Foul'])) & (df['PLAYER_ID'].isnull())]
    for idx, row in missing_data.iterrows():
        print(f"[ATTRIBUTION ERROR] Q{row['QUARTER']} {row['TIME_REMAINING']} - {row['EVENT_TYPE']} is missing a Player ID.")
        errors += 1
    return errors

In [8]:
def run_live_audit(dataframe):
    print_header("NIGHTLY STATS AUDIT REPORT")

    total_errors = 0
    total_errors += audit_scoring_values(dataframe)
    total_errors += audit_rebound_logic(dataframe)
    total_errors += audit_missing_attributions(dataframe)

    print("\n" + "-"*40)
    if total_errors == 0:
        print("STATUS: GREEN - All data lines verified.")
    else:
        print(f"STATUS: RED - {total_errors} discrepancies flagged for manual review.")
    print("-"*40)

# Execute the audit
run_live_audit(df)


NIGHTLY STATS AUDIT REPORT
[SCORING ERROR] Q1 09:15 - Free Throw logged as 2 points (Player: 1003.0)
[SEQUENCE ERROR] Q1 11:42 - Rebound logged immediately after a Made Field Goal.
[ATTRIBUTION ERROR] Q1 10:30 - Turnover is missing a Player ID.

----------------------------------------
STATUS: RED - 3 discrepancies flagged for manual review.
----------------------------------------
